In [3]:
import rasterio as rio
from skimage.segmentation import quickshift
import numpy as np
import geopandas as gpd
from rasterio.features import shapes
from src.mslandcover.inference import parallel_quickshift
from src.mslandcover.config import MSTM_PROJ4
from tqdm import tqdm

In [4]:
lc_probs_raster_path = './data/inference_results/149/lc_probs_clipped.tif'
raster_path = r"G:\mslc_inf_test\starkville_msu_2023.tif"

census_ms_places_shp_path = './data/shapefiles/tl_2024_28_place/tl_2024_28_place.shp'
ms_places_gdf = gpd.read_file(census_ms_places_shp_path)
starville_msu_gdf = ms_places_gdf[ms_places_gdf['NAME'].isin(['Starkville', 'Mississippi State'])]

starville_msu_reproj_gdf = starville_msu_gdf.to_crs(MSTM_PROJ4)
try:
    starville_msu_geom = starville_msu_reproj_gdf.union_all()
except:
    starville_msu_geom = starville_msu_reproj_gdf.unary_union


with rio.open(raster_path) as src:
    raster_data = src.read()
    profile = src.profile

with rio.open(lc_probs_raster_path) as src:
    lc_probs_raster = src.read()

segments = parallel_quickshift(raster_data)

In [3]:
post_processes_lc_preds = np.zeros_like(segments, dtype=np.uint8)
unique_segments = np.unique(segments)


In [6]:
from functools import partial
import multiprocessing as mp
from multiprocessing.shared_memory import SharedMemory
from tqdm import tqdm
import os

def create_shared_memory(array):
    shm = SharedMemory(create=True, size=array.nbytes)
    shm_buf = np.ndarray(array.shape, dtype=array.dtype, buffer=shm.buf)
    np.copyto(shm_buf, array)
    return shm, shm_buf

def cleanup_shared_memory(*shms):
    for shm in shms:
        shm.close()
        shm.unlink()

def process_segment(segment, segments_shm_buf, lc_probs_raster_shm_buf, post_processes_lc_preds_shm_buf, pbar=None):
    mask = segments_shm_buf == segment
    post_processes_lc_preds_shm_buf[mask] = np.argmax(np.mean(lc_probs_raster_shm_buf[:, mask], axis=1))
    if pbar is not None:
        pbar.update(1)

def process_segments(segments, lc_probs_raster, num_processes=mp.cpu_count()):
    post_processes_lc_preds = np.zeros_like(segments, dtype=np.uint8)
    unique_segments = np.unique(segments)


    segments_shm, segments_shm_buf = create_shared_memory(segments)
    lc_probs_raster_shm, lc_probs_raster_shm_buf = create_shared_memory(lc_probs_raster)
    post_processes_lc_preds_shm, post_processes_lc_preds_shm_buf = create_shared_memory(post_processes_lc_preds)

    with tqdm(total=len(unique_segments), desc="Processing segments", unit="segments") as pbar:
        worker = partial(process_segment, segments_shm_buf=segments_shm_buf, lc_probs_raster_shm_buf=lc_probs_raster_shm_buf, post_processes_lc_preds_shm_buf=post_processes_lc_preds_shm_buf, pbar=pbar)
        with mp.Pool(num_processes) as pool:
            pool.map(worker, unique_segments)

    np.copyto(post_processes_lc_preds, post_processes_lc_preds_shm_buf)

    cleanup_shared_memory(segments_shm, lc_probs_raster_shm, post_processes_lc_preds_shm)

    return post_processes_lc_preds

# def process_segments(segments, lc_probs_raster, num_processes=mp.cpu_count()):
#     # Use numpy.unique with return_index=True to get segment positions
#     unique_segments, segment_indices = np.unique(segments, return_inverse=True)
    
#     # Pre-allocate output array
#     post_processes_lc_preds = np.zeros_like(segments, dtype=np.uint8)
    
#     # Process in chunks to reduce memory pressure
#     chunk_size = len(unique_segments) // num_processes
#     with tqdm(total=len(unique_segments), desc="Processing segments", unit="segments") as pbar:
#         for chunk_start in range(0, len(unique_segments), chunk_size):
#             chunk_end = min(chunk_start + chunk_size, len(unique_segments))
#             chunk_segments = unique_segments[chunk_start:chunk_end]
            
#             # Create masks for this chunk all at once
#             chunk_masks = segment_indices == chunk_segments[:, None]
            
#             # Process probabilities for all segments in chunk at once
#             chunk_means = np.mean(lc_probs_raster[:, chunk_masks], axis=1)
#             chunk_predictions = np.argmax(chunk_means, axis=0)
            
#             # Assign results
#             for i, segment in enumerate(chunk_segments):
#                 post_processes_lc_preds[segments == segment] = chunk_predictions[i]
            
#             pbar.update(len(chunk_segments))
            
#     return post_processes_lc_preds

post_processed_lc_preds = process_segments(segments, lc_probs_raster, num_processes=4)

Processing segments:   0%|          | 0/65536 [01:31<?, ?segments/s]


TypeError: cannot pickle '_hashlib.HMAC' object